# `wf_transaction_history` 단계별 Testbed

계좌 후보가 여러 개인 거래내역 조회를 실행해 Webhook 중단과 검증된 Resume까지 확인한다.

## 전체 흐름

```text
사용자 발화 → 계좌 확인 API → 계좌 선택 Webhook → 중단
→ Backend 검증 Resume → 기간 기본값 적용 → 거래내역 API → 결과 Webhook
```

Resume 이후 Agent는 계좌를 다시 조회하지 않는다.

In [ ]:
import json
from datetime import date, datetime, timezone

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing.mock_backend import MockBackend
from agent.testing.transaction_history import create_transaction_history_mock_testbed
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.query_slot_extraction import extract_transaction_slots_llm_first
from agent.workflows.transaction_history import extract_transaction_slots_from_text


def show(title, value):
    print(f"\n--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


NOW = datetime(2026, 7, 19, 3, 0, tzinfo=timezone.utc)
config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("test-token"),
    agent_webhook_secret=SecretStr("test-secret"),
    retry_backoff_seconds=0,
)
print("설정 완료")

## 1. 계약과 LLM 우선 Slot 추출 확인

LLM은 비정형 발화의 의미를 구조화하고, 기간 날짜 계산은 코드가 수행한다. 계좌와 가맹점 표현은 사용자 원문에 있는 구절만 사용한다.

In [ ]:
workflow = WorkflowContractStore().get_workflow("wf_transaction_history")
show("Step 계약", [
    {"step_id": step["step_id"], "mode": step["interaction_mode"], "contract_id": step.get("contract_id")}
    for step in workflow["steps"]
])
message = "최근 거래내역 보여줘"
requested_date = date(2026, 7, 19)
show("LLM 우선 추출 결과", await extract_transaction_slots_llm_first(message, requested_date))
show("결정적 폴백 참고", extract_transaction_slots_from_text(message, requested_date))

## 2. 계좌 선택까지의 Mock 응답 준비

In [ ]:
def account(account_id, alias):
    return {
        "account_id": account_id,
        "bank_name": "토스뱅크",
        "account_alias": alias,
        "account_type": "checking",
        "masked_account_number": "1000-***-1234",
        "currency": "KRW",
        "is_default": account_id == "acc_001",
        "status": "active",
    }


backend = MockBackend()
backend.add_success("GET", "/api/v1/agent-tools/accounts", {
    "account_resolution_outcome": "selection_required",
    "accounts": [account("acc_001", "생활비"), account("acc_002", "용돈")],
    "account_ids": [],
})
backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_select"})
backend.add_success("POST", "/api/v1/agent-tools/transactions:query", {
    "transaction_results": [{
        "transaction_id": "txn_001",
        "account_id": "acc_002",
        "account_alias": "용돈",
        "occurred_at": "2026-07-18T12:30:00+09:00",
        "transaction_type": "card_payment",
        "amount": 18500,
        "currency": "KRW",
        "transaction_title": "배민",
        "category": "식비",
    }],
    "transaction_query_id": "txq_001",
    "next_cursor": None,
})
backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_result"})

## 3. Workflow 시작과 중단 확인

In [ ]:
testbed = create_transaction_history_mock_testbed(
    backend,
    config,
    thread_id="thread_transactions",
    input_request_ids=["input_account_001"],
    now=NOW,
)
waiting = await testbed.start(
    message="최근 거래내역 보여줘",
    request_id="req_transactions",
    chat_session_id="chat_001",
    execution_context_id="exec_001",
)
show("중단 결과", waiting.model_dump(mode="json"))
show("계좌 선택 Webhook", testbed.webhook_events(include_payload=True))

## 4. Backend가 검증한 계좌 선택값으로 Resume

In [ ]:
completed = await testbed.resume_input(
    agent_thread_id="thread_transactions",
    request_id="req_transactions_resume",
    chat_session_id="chat_001",
    execution_context_id="exec_001",
    input_request_id="input_account_001",
    value={"account_selection_outcome": "selected", "account_ids": ["acc_002"]},
)
show("완료 결과", completed.model_dump(mode="json"))
show("전체 HTTP Timeline", backend.exchange_timeline(include_payload=True))
show("최종 State", await testbed.state("thread_transactions"))
await testbed.aclose()